# Assignment 1: Image Classification (ANN vs. CNN)

**Student:** Rishi

Compare a dense **ANN (MLP)** with a **CNN** on the **Intel Image Classification** dataset (~25,000 images, six scene categories: buildings, forest, glacier, mountain, sea, street).

All images are resized to **150×150**, normalized, and evaluated with a fair pipeline: full dataset, stratified validation split, capacity-matched models, and complete test-set metrics.


## Before running

1. Install dependencies: pip install -r ../requirements.txt
2. Place the Intel dataset in ../data/raw/ (or run python ../download_dataset.py).
3. Run all cells top-to-bottom. Saved checkpoints in ../results/models/ allow faster re-runs without retraining.


## 2. Dataset Procurement

**Recommended Dataset:** Intel Image Classification (Kaggle).

**Description:** Approximately **25,000 images** across six natural categories: `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`.

This project uses the complete supplied copy (**24,335 raw images**):
- **14,034** labeled training images (`seg_train`)
- **3,000** labeled test images (`seg_test`)
- **7,301** unlabeled prediction images (`seg_pred`, counted but not used for supervised training)

Supervised training uses **every labeled training image** with a **seeded stratified train/validation split**. Final evaluation uses the **complete official test split** (all 3,000 images, all six classes).


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from model_architectures.ann_classifier import ANNClassifier
from model_architectures.cnn_classifier import CNNClassifier
from project_utils.data import (
    EXPECTED_CLASSES,
    build_dataloaders,
    class_counts_from_loader,
    get_raw_sample_images,
    get_transforms,
    load_image_for_display,
    summarize_dataset_inventory,
)
from project_utils.evaluation import evaluate_model, print_evaluation_summary
from project_utils.plots import (
    plot_class_distribution,
    plot_confusion_matrix,
    plot_predictions,
    plot_sample_images,
    plot_training_curves,
    print_comparison_table,
    save_comparison_table,
)
from project_utils.prediction import predict_image
from project_utils.training import (
    count_parameters,
    get_device_info,
    load_training_histories,
    print_model_summary,
    set_seed,
    train_model,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")


Project root: C:\Users\siddh\Downloads\ANN-vs-CNN-main\ANN-vs-CNN-main\scene_image_classifier
PyTorch version: 2.13.0+cpu


## 3.1 Data Preprocessing

- Load raw images from the Intel Image Classification dataset (~25,000 images, six categories).
- Resize every image to **150×150** pixels.
- Normalize pixel values (ImageNet channel statistics).
- Use a **seeded stratified train/validation split** on the full **14,034** labeled training images so all six classes remain represented.
- Evaluate on the **complete official test split** (**3,000** images, all six classes).


In [2]:
DATASET_ROOT = PROJECT_ROOT / "data" / "raw"

RESULTS_ROOT = PROJECT_ROOT / "results"
MODEL_DIR = RESULTS_ROOT / "models"
FIGURE_DIR = RESULTS_ROOT / "figures"
METRICS_DIR = RESULTS_ROOT / "metrics"
PREDICTION_DIR = RESULTS_ROOT / "predictions"

for directory in [MODEL_DIR, FIGURE_DIR, METRICS_DIR, PREDICTION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 150
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
RANDOM_SEED = 42
VAL_RATIO = 0.2
PATIENCE = 3
NUM_WORKERS = 0

set_seed(RANDOM_SEED)
device, device_info = get_device_info()
inventory = summarize_dataset_inventory(DATASET_ROOT)

print(device_info)
print(f"Dataset root: {DATASET_ROOT}")
print(f"Total raw images: {inventory['total_raw_images']:,}")
print(f"Labeled train images: {inventory['labeled_train_images']:,}")
print(f"Labeled test images: {inventory['labeled_test_images']:,}")
print(f"Unlabeled prediction images: {inventory['unlabeled_prediction_images']:,}")
print(f"Image size: {IMAGE_SIZE} x {IMAGE_SIZE}")
print(f"Reproducibility seed: {RANDOM_SEED}")


CUDA not available. Using CPU.
Dataset root: C:\Users\siddh\Downloads\ANN-vs-CNN-main\ANN-vs-CNN-main\scene_image_classifier\data\raw
Total raw images: 24,335
Labeled train images: 14,034
Labeled test images: 3,000
Unlabeled prediction images: 7,301
Image size: 150 x 150
Reproducibility seed: 42


In [ ]:
ann_model = ANNClassifier(
    num_classes=len(class_names),
    input_channels=3,
    image_size=IMAGE_SIZE,
    hidden_sizes=(160, 32),  # capacity-matched to CNN (~10M params), not 34.7M
    dropout=0.5,
)
cnn_model = CNNClassifier(
    num_classes=len(class_names),
    input_channels=3,
    image_size=IMAGE_SIZE,
    dropout=0.5,
)

ann_params = print_model_summary(ann_model, "ANN")
cnn_params = print_model_summary(cnn_model, "CNN")
print(
    f"Comparable capacity: ANN={ann_params['trainable']:,}, "
    f"CNN={cnn_params['trainable']:,} trainable parameters "
    f"(ratio {ann_params['trainable'] / cnn_params['trainable']:.2f}x)"
)
assert ann_params["trainable"] < 15_000_000, "ANN is too large versus the CNN."

In [ ]:
HISTORY_PATH = METRICS_DIR / "training_histories.json"
CHECKPOINTS_EXIST = (
    (MODEL_DIR / "best_ann_model.pth").is_file()
    and (MODEL_DIR / "best_cnn_model.pth").is_file()
    and HISTORY_PATH.is_file()
)

if CHECKPOINTS_EXIST:
    print("Saved checkpoints found — loading models and training history.")
    histories = load_training_histories(HISTORY_PATH)
    ann_history = histories["ann"]
    cnn_history = histories["cnn"]
    ann_model.load_state_dict(
        torch.load(MODEL_DIR / "best_ann_model.pth", map_location=device)
    )
    cnn_model.load_state_dict(
        torch.load(MODEL_DIR / "best_cnn_model.pth", map_location=device)
    )
    ann_model.to(device)
    cnn_model.to(device)
else:
    print("Training both models from scratch...")
    set_seed(RANDOM_SEED)
    ann_history, ann_model = train_model(
        ann_model,
        train_loader,
        val_loader,
        device,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        patience=PATIENCE,
        model_save_path=MODEL_DIR / "best_ann_model.pth",
    )

    set_seed(RANDOM_SEED)
    cnn_history, cnn_model = train_model(
        cnn_model,
        train_loader,
        val_loader,
        device,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        patience=PATIENCE,
        model_save_path=MODEL_DIR / "best_cnn_model.pth",
    )

    print(f"\nANN training time: {ann_history.total_training_time_sec:.2f} s")
    print(f"CNN training time: {cnn_history.total_training_time_sec:.2f} s")
    print(f"ANN best validation accuracy: {ann_history.best_val_accuracy:.4f}")
    print(f"CNN best validation accuracy: {cnn_history.best_val_accuracy:.4f}")
    save_training_histories(
        {"ann": ann_history, "cnn": cnn_history},
        METRICS_DIR / "training_histories.json",
    )

eval_loader = test_loader
eval_split = "test"
assert len(eval_loader.dataset) >= 3000
print(f"Evaluating every image in the {eval_split} split ({len(eval_loader.dataset):,} images).")
print("No first-1,000 restriction. Confusion matrix uses all six class labels.")

ann_results = evaluate_model(ann_model, eval_loader, device, class_names)
cnn_results = evaluate_model(cnn_model, eval_loader, device, class_names)
print_evaluation_summary(ann_results, "ANN", eval_split)
print_evaluation_summary(cnn_results, "CNN", eval_split)
assert ann_results.confusion_matrix.shape == (6, 6)
assert cnn_results.confusion_matrix.shape == (6, 6)

for split_name, counts in split_counts.items():
    plot_class_distribution(
        counts,
        f"{split_name.title()} class distribution",
        FIGURE_DIR / f"{split_name}_class_distribution.png",
    )
plot_training_curves(ann_history, "ANN", FIGURE_DIR)
plot_training_curves(cnn_history, "CNN", FIGURE_DIR)
plot_confusion_matrix(
    ann_results.confusion_matrix,
    class_names,
    "ANN confusion matrix — complete test set",
    FIGURE_DIR / "ann_confusion_matrix.png",
)
plot_confusion_matrix(
    cnn_results.confusion_matrix,
    class_names,
    "CNN confusion matrix — complete test set",
    FIGURE_DIR / "cnn_confusion_matrix.png",
)

(METRICS_DIR / "ann_classification_report.txt").write_text(ann_results.classification_report)
(METRICS_DIR / "cnn_classification_report.txt").write_text(cnn_results.classification_report)

comparison_rows = [
    {
        "Model": "ANN",
        "Trainable Parameters": ann_params["trainable"],
        "Training Time (seconds)": ann_history.total_training_time_sec,
        "Best Validation Accuracy": ann_history.best_val_accuracy,
        "Evaluation Images": len(eval_loader.dataset),
        "Test Accuracy": ann_results.accuracy,
        "Macro Precision": ann_results.macro_precision,
        "Macro Recall": ann_results.macro_recall,
        "Macro F1": ann_results.macro_f1,
        "Weighted F1": ann_results.weighted_f1,
    },
    {
        "Model": "CNN",
        "Trainable Parameters": cnn_params["trainable"],
        "Training Time (seconds)": cnn_history.total_training_time_sec,
        "Best Validation Accuracy": cnn_history.best_val_accuracy,
        "Evaluation Images": len(eval_loader.dataset),
        "Test Accuracy": cnn_results.accuracy,
        "Macro Precision": cnn_results.macro_precision,
        "Macro Recall": cnn_results.macro_recall,
        "Macro F1": cnn_results.macro_f1,
        "Weighted F1": cnn_results.weighted_f1,
    },
]
comparison_df = save_comparison_table(comparison_rows, METRICS_DIR / "model_comparison.csv")
display(comparison_df)
print_comparison_table(comparison_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
comparison_df.plot.bar(
    x="Model",
    y="Best Validation Accuracy",
    ax=axes[0],
    legend=False,
    color=["#6b7280", "#2563eb"],
    title="Validation accuracy",
)
comparison_df.plot.bar(
    x="Model",
    y="Training Time (seconds)",
    ax=axes[1],
    legend=False,
    color=["#6b7280", "#2563eb"],
    title="Training time (seconds)",
)
comparison_df.plot.bar(
    x="Model",
    y="Trainable Parameters",
    ax=axes[2],
    legend=False,
    color=["#6b7280", "#2563eb"],
    title="Trainable parameters",
)
for axis in axes:
    axis.set_xlabel("")
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "ann_cnn_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ANN/CNN capacity ratio: {ann_params['trainable'] / cnn_params['trainable']:.2f}x")
print(f"Validation accuracy: ANN={ann_history.best_val_accuracy:.4f}, CNN={cnn_history.best_val_accuracy:.4f}")
print(f"Test accuracy: ANN={ann_results.accuracy:.4f}, CNN={cnn_results.accuracy:.4f}")

## Run the experiment

The cells below load saved checkpoints when available (otherwise train from scratch), evaluate on all 3,000 test images, and save figures/metrics to ../results/.

## 3.3 Comparison Analysis

Both models share the same:
- labeled training pool (full seg_train split),
- batch size, optimizer (Adam), learning rate, epochs, and early stopping,
- evaluation procedure on the full test set.

We compare **training time**, **validation accuracy**, and **total trainable parameters**, plus test accuracy and macro/weighted classification metrics.

In [3]:
loaders = build_dataloaders(
    dataset_root=DATASET_ROOT,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    val_ratio=VAL_RATIO,
    seed=RANDOM_SEED,
    num_workers=NUM_WORKERS,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]
class_names = loaders["class_names"]

required_classes = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
assert sorted(class_names) == required_classes
assert test_loader is not None, "Complete official test split is required."
assert len(train_loader.dataset) + len(val_loader.dataset) >= 14000
assert len(test_loader.dataset) >= 3000, "Must evaluate the full test set, not the first 1,000 images."

split_counts = {
    "train": class_counts_from_loader(train_loader),
    "validation": class_counts_from_loader(val_loader),
    "test": class_counts_from_loader(test_loader),
}

class_distribution = pd.DataFrame(split_counts).reindex(required_classes)
assert (class_distribution > 0).all().all()
display(class_distribution)
print(class_distribution.sum().rename("Total images"))
print("Train+val uses the full labeled Intel training split (no 2,000-image cap).")
print(f"Test images evaluated: {len(test_loader.dataset):,} (complete official test set).")
print("Split method: seeded stratified split (seed=42), not random_split().")
print("All six classes are present in train, validation, and test.")


,train,validation,test
buildings,1753,438,437
forest,1817,454,474
glacier,1923,481,553
mountain,2010,502,525
sea,1819,455,510
street,1906,476,501


train         11228
validation     2806
test           3000
Name: Total images, dtype: int64
All six classes are present in train, validation, and test.


In [ ]:
HISTORY_PATH = METRICS_DIR / "training_histories.json"
CHECKPOINTS_EXIST = (
    (MODEL_DIR / "best_ann_model.pth").is_file()
    and (MODEL_DIR / "best_cnn_model.pth").is_file()
    and HISTORY_PATH.is_file()
)

if CHECKPOINTS_EXIST:
    print("Saved checkpoints found — loading models and training history.")
    histories = load_training_histories(HISTORY_PATH)
    ann_history = histories["ann"]
    cnn_history = histories["cnn"]
    ann_model.load_state_dict(
        torch.load(MODEL_DIR / "best_ann_model.pth", map_location=device)
    )
    cnn_model.load_state_dict(
        torch.load(MODEL_DIR / "best_cnn_model.pth", map_location=device)
    )
    ann_model.to(device)
    cnn_model.to(device)
else:
    print("Training both models from scratch...")
    set_seed(RANDOM_SEED)
    ann_history, ann_model = train_model(
        ann_model,
        train_loader,
        val_loader,
        device,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        patience=PATIENCE,
        model_save_path=MODEL_DIR / "best_ann_model.pth",
    )

    set_seed(RANDOM_SEED)
    cnn_history, cnn_model = train_model(
        cnn_model,
        train_loader,
        val_loader,
        device,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        patience=PATIENCE,
        model_save_path=MODEL_DIR / "best_cnn_model.pth",
    )

    print(f"\nANN training time: {ann_history.total_training_time_sec:.2f} s")
    print(f"CNN training time: {cnn_history.total_training_time_sec:.2f} s")
    print(f"ANN best validation accuracy: {ann_history.best_val_accuracy:.4f}")
    print(f"CNN best validation accuracy: {cnn_history.best_val_accuracy:.4f}")
    save_training_histories(
        {"ann": ann_history, "cnn": cnn_history},
        METRICS_DIR / "training_histories.json",
    )

eval_loader = test_loader
eval_split = "test"
assert len(eval_loader.dataset) >= 3000
print(f"Evaluating every image in the {eval_split} split ({len(eval_loader.dataset):,} images).")
print("No first-1,000 restriction. Confusion matrix uses all six class labels.")

ann_results = evaluate_model(ann_model, eval_loader, device, class_names)
cnn_results = evaluate_model(cnn_model, eval_loader, device, class_names)
print_evaluation_summary(ann_results, "ANN", eval_split)
print_evaluation_summary(cnn_results, "CNN", eval_split)
assert ann_results.confusion_matrix.shape == (6, 6)
assert cnn_results.confusion_matrix.shape == (6, 6)

for split_name, counts in split_counts.items():
    plot_class_distribution(
        counts,
        f"{split_name.title()} class distribution",
        FIGURE_DIR / f"{split_name}_class_distribution.png",
    )
plot_training_curves(ann_history, "ANN", FIGURE_DIR)
plot_training_curves(cnn_history, "CNN", FIGURE_DIR)
plot_confusion_matrix(
    ann_results.confusion_matrix,
    class_names,
    "ANN confusion matrix — complete test set",
    FIGURE_DIR / "ann_confusion_matrix.png",
)
plot_confusion_matrix(
    cnn_results.confusion_matrix,
    class_names,
    "CNN confusion matrix — complete test set",
    FIGURE_DIR / "cnn_confusion_matrix.png",
)

(METRICS_DIR / "ann_classification_report.txt").write_text(ann_results.classification_report)
(METRICS_DIR / "cnn_classification_report.txt").write_text(cnn_results.classification_report)

comparison_rows = [
    {
        "Model": "ANN",
        "Trainable Parameters": ann_params["trainable"],
        "Training Time (seconds)": ann_history.total_training_time_sec,
        "Best Validation Accuracy": ann_history.best_val_accuracy,
        "Evaluation Images": len(eval_loader.dataset),
        "Test Accuracy": ann_results.accuracy,
        "Macro Precision": ann_results.macro_precision,
        "Macro Recall": ann_results.macro_recall,
        "Macro F1": ann_results.macro_f1,
        "Weighted F1": ann_results.weighted_f1,
    },
    {
        "Model": "CNN",
        "Trainable Parameters": cnn_params["trainable"],
        "Training Time (seconds)": cnn_history.total_training_time_sec,
        "Best Validation Accuracy": cnn_history.best_val_accuracy,
        "Evaluation Images": len(eval_loader.dataset),
        "Test Accuracy": cnn_results.accuracy,
        "Macro Precision": cnn_results.macro_precision,
        "Macro Recall": cnn_results.macro_recall,
        "Macro F1": cnn_results.macro_f1,
        "Weighted F1": cnn_results.weighted_f1,
    },
]
comparison_df = save_comparison_table(comparison_rows, METRICS_DIR / "model_comparison.csv")
display(comparison_df)
print_comparison_table(comparison_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
comparison_df.plot.bar(
    x="Model",
    y="Best Validation Accuracy",
    ax=axes[0],
    legend=False,
    color=["#6b7280", "#2563eb"],
    title="Validation accuracy",
)
comparison_df.plot.bar(
    x="Model",
    y="Training Time (seconds)",
    ax=axes[1],
    legend=False,
    color=["#6b7280", "#2563eb"],
    title="Training time (seconds)",
)
comparison_df.plot.bar(
    x="Model",
    y="Trainable Parameters",
    ax=axes[2],
    legend=False,
    color=["#6b7280", "#2563eb"],
    title="Trainable parameters",
)
for axis in axes:
    axis.set_xlabel("")
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "ann_cnn_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ANN/CNN capacity ratio: {ann_params['trainable'] / cnn_params['trainable']:.2f}x")
print(f"Validation accuracy: ANN={ann_history.best_val_accuracy:.4f}, CNN={cnn_history.best_val_accuracy:.4f}")
print(f"Test accuracy: ANN={ann_results.accuracy:.4f}, CNN={cnn_results.accuracy:.4f}")

In [ ]:
batch_images, batch_labels = next(iter(eval_loader))
with torch.no_grad():
    batch_predictions = cnn_model(batch_images.to(device)).argmax(dim=1).cpu()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
shown_images = [(image * std + mean).permute(1, 2, 0).clamp(0, 1).numpy() for image in batch_images[:8]]
shown_true = [class_names[label] for label in batch_labels[:8]]
shown_predicted = [class_names[label] for label in batch_predictions[:8]]
plot_predictions(
    shown_images,
    shown_true,
    shown_predicted,
    "CNN predictions",
    PREDICTION_DIR / "cnn_predictions.png",
)
print(f"Saved experiment artifacts under: {RESULTS_ROOT}")

In [ ]:
IMAGE_PATH = None

if IMAGE_PATH:
    prediction = predict_image(
        IMAGE_PATH,
        cnn_model,
        get_transforms(image_size=IMAGE_SIZE),
        class_names,
        device,
    )
    image = load_image_for_display(Path(IMAGE_PATH))
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"{prediction['class_name'].upper()} ({prediction['confidence']:.2%})")
    plt.show()
    probabilities = pd.Series(prediction["probabilities"])
    probabilities.sort_values().plot.barh(figsize=(7, 4), color="steelblue")
    plt.xlabel("Probability")
    plt.title("CNN class probabilities")
    plt.tight_layout()
    plt.show()
else:
    print("Set IMAGE_PATH to run single-image prediction.")